## Tarea: Modelos ARMA


- Objetivo: Ajustar modelos ARMA a una serie de tiempo.
- Tipo de actividad: Individual
- Tipo de evaluación: Sumativa 
- Puntaje: 100 puntos
- Calificación: Escala de 1 a 7, con una exigencia de 50%. La nota mínima para aprobar es 4.0.

## Introducción
En esta tarea, nos enfocaremos en el análisis de series temporales mediante la descomposición de la serie, el ajuste de modelos de regresión a la tendencia, y la modelación de los residuos mediante modelos ARMA. El objetivo es comprender cómo estos componentes pueden combinarse para mejorar la predicción de observaciones futuras y evaluar la calidad del ajuste obtenido.

## Descripción del problema
La serie IPC.xlsx corresponde al Índice de Precios al Consumidor de Chile desde abril de 1989 hasta junio de 2023, medido mensualmente. El objetivo del análisis es predecir el IPC para el segundo semestre de 2023, es decir, de julio a diciembre de 2023.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
from darts import TimeSeries
import matplotlib.pyplot as plt
import statsmodels.api as sm

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from statsmodels.tsa.seasonal import seasonal_decompose

from sklearn.metrics import mean_squared_error

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA

import warnings
warnings.filterwarnings('ignore')


Utiliza el código proporcionado a continuación para realizar el análisis y las predicciones requeridas. No es necesario modificar el código; simplemente ejecútalo y utiliza los resultados en tu análisis.

In [ ]:
df = pd.read_excel('ipc.xlsx')

df['Fecha'] = pd.to_datetime(df[['YEAR', 'MONTH']].assign(day=1))

df.set_index('Fecha', inplace=True)
df.drop(['YEAR', 'MONTH'], axis=1, inplace=True)

ipc = TimeSeries.from_series(df['IPC'])

ipc_train, ipc_val = ipc.split_before(pd.Timestamp('2023-01-01'))
train_data = ipc_train.pd_series()

decomposition_result = seasonal_decompose(train_data, model='additive', period=12, extrapolate_trend='freq')

trend = decomposition_result.trend.dropna()
seasonal = decomposition_result.seasonal.dropna()
residual = decomposition_result.resid.dropna()

X = np.arange(len(trend)).reshape(-1, 1)
y = trend.values

poly = PolynomialFeatures(degree=3)
X_poly = poly.fit_transform(X)

regression_model = LinearRegression()
regression_model.fit(X_poly, y)

dates_jan_jun_2023 = pd.date_range(start='2023-01-01', end='2023-06-01', freq='MS')
X_future = np.arange(len(trend), len(trend) + len(dates_jan_jun_2023)).reshape(-1, 1)
X_future_poly = poly.transform(X_future)

predicted_trend = regression_model.predict(X_future_poly)

seasonality_by_month = seasonal.groupby(seasonal.index.month).mean()
months = [1, 2, 3, 4, 5, 6]
future_seasonality = seasonality_by_month.loc[months].values.flatten()

forecast_without_residuals = predicted_trend + future_seasonality
forecast_without_residuals_series = pd.Series(forecast_without_residuals, index=dates_jan_jun_2023)


In [ ]:
## Gráfica de la componente de ruido a la cual se debe ajustar un modelo ARMA
plt.figure(figsize=(10, 4))
plt.plot(residual, label='Residuos')
plt.title('Componente de ruido en la descompocisión Aditiva - IPC')
plt.xlabel('Fecha')
plt.legend()
plt.show()


### Pregunta 1

Realice la prueba de Ljung-Box con `lags=10` para la serie `residual`. Guarde el resultado del test en el objeto `lb_test` y el p-valor en `p_value`.

Use la función `acorr_ljungbox`con los argumentos `lags=[10]` y `return_df=True`, es decir,

* `acorr_ljungbox(serie, lags=[10], return_df=True)`

**Obs**: con `lags=[10]` se calcula la prueba solo para 10, mientras que con `lags=10` se calcula la prueba desde 1 a 10.

In [ ]:
# Guardar la información:
# lb_test
# p_value

# your code here
raise NotImplementedError

In [ ]:
# Prueba oculta P1: Check de la prueba de Ljung-Box

### Pregunta 2

Indica la interpretación del test de Ljung-Box en el objeto `respuesta_ljungbox`:

   * respuesta_ljungbox = 'Si, existe autocorrelación significativa en los residuos'
   * respuesta_ljungbox = 'No, no existe autocorrelación significativa en los residuos'

In [ ]:
# Guardar la información:
# respuesta_ljungbox

# Realizar prueba de Ljung-Box en los residuos
# your code here
raise NotImplementedError


In [ ]:
# Prueba oculta P2: Check de la interpretación de la prueba de Ljung-Box

### Pregunta 3 

Realice la descomposición estacional de la serie `residual` usando un modelo aditivo. Guarde la serie en el objeto `residual_seasonal`. Grafique las componentes de la descomposición y comente si existe evidencia visual de estacionalidad de algún tipo.

Observación: El comentario sobre la serie debe guardarlo en el objeto `comentario_serie`. Incluya strings del tipo 'si existe' o 'no existe', para que sea corregido adecuadamente (si existe, escriba Si sin tilde)

In [ ]:
## Realizar la descomposición de la serie y comentar
# your code here
raise NotImplementedError

In [ ]:
# Prueba oculta P3: Check de la interpretación de la estacionalidad
###BEGIN HIDDEN TEST
assert residual_seasonal is not None, "residual_seasonal no se generó."
assert isinstance(comentario_estacionalidad, str), "comentario_estacionalidad debe ser un string."
assert "si" in comentario_estacionalidad.lower(), "No es correcta la interpretación."
###END HIDDEN TEST

### Pregunta 4

Grafique las funciones de autocorrelación y de autocorrelación parcial. Proponga, a partir de estas, tres modelos ARMA para la componente `residual` e indique cual de ellos escoger mediante el AIC. Guarde cada modelo con el formato:

`arma_model1 = ARIMA(residual, order=(p,0,q))`

`arma_results1 = arma_model1.fit()`

cambiando el 1 por 2 y 3 para los otros modelos. Guarde el `AIC` en el objeto

`AIC = [arma_results1.aic, arma_results2.aic, arma_results3.aic]`

Guarde el modelo escogido en `mod_select` indicando la posición del mejor modelo, es decir,
* `mod_select=0`
* `mod_select=1`
* `mod_select=2`

In [ ]:
# Guardar la información:
# arma_model1 
# arma_model2
# arma_model3

# arma_results1
# arma_results2
# arma_results3

# AIC
# mod_select

# your code here
raise NotImplementedError

In [ ]:
# Prueba oculta P4: Check de la selección de modelos mediante AIC

### Pregunta 5

Realice predicciones para los valores del conjunto de validación para los datos de IPC. Para ello, realizar la predicción del ruido con el mejor modelo según el AIC anterior y guardarlos en 

`residual_forecast = arma_results.forecast(steps=len(dates_jan_jun_2023))`,

luego agregar estas predicciones a las almacenadas en **forecast_without_residuals**.

Guarde el orden del modelo final ajustado en `arma_order` y el error cuadrático medio (MSE) en `mse`.

El formato para guardar el orden del modelo es `arma_order = (p,0,q)`, el cual puede ser obtenido directamente desde
* `arma_order = arma_model.order`, donde `arma_model` es el mejor modelo seleccionado con el AIC.



In [ ]:
# Guardar la información:
# arma_order = (p,0,q) 
# mse
# residual_forecast 

# your code here
raise NotImplementedError


In [ ]:
# Prueba oculta P5: Check de las predicciones del modelo ARMA


### Pregunta 6

Realice nuevamente las predicciones para los valores del conjunto de validación para los datos de IPC. Ahora debe buscar un modelo de la familia ARMA para los residuos con el objetivo de obtener un error cuadrático medio (MSE) menor a 80. Es decir, el objetivo es seleccionar el modelo que minimice el MSE. (Este modelo podria coincidir o no con el modelo escogido por el AIC).

Guarde el resultado del MSE en `mse_final`.

Se sugiere volver a revisar las gráficas de acf y pacf para proponer otros modelos en caso de ser necesario.

In [ ]:
# Guardar la información:
# mse_final

# your code here
raise NotImplementedError


In [ ]:
# Prueba oculta P6: Check del MSE final